# 로컬 M1 합성 CPU 통합 진단
테스트 세션이 고정한 소스와 ready notebook으로 CPU에서 한 번만 실행합니다. 성공·실패·timeout 모두 마지막 결과 회수 셀을 실행하세요. 실제 데이터·GPU 성능 검증은 아닙니다. 자동 설치·Drive 연결·재시도는 없습니다.


In [ ]:
# 테스트 세션에서 고정한 ready notebook만 실행한다. 자동 설치·Drive·재시도 없음.
import sys, os, io, json, stat, re, hashlib, tempfile, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

EXPECTED_REVISION = ""
EXPECTED_SOURCE_SHA256 = ""
if not re.fullmatch(r"[0-9a-f]{40}", EXPECTED_REVISION) or not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SOURCE_SHA256):
    raise RuntimeError("테스트 세션의 고정 revision/source ZIP SHA256을 설정한 ready notebook이 필요합니다.")
if "M1_RUN_ROOT" in globals():
    raise RuntimeError("이미 준비한 실행입니다. 기존 결과를 회수하고 중단하세요.")
M1_RUN_ROOT = Path(tempfile.mkdtemp(prefix="rewind_m1_smoke_", dir="/content"))
SOURCE_OK = False
receipt = {"status":"failed", "expected_revision":EXPECTED_REVISION, "expected_zip_sha256":EXPECTED_SOURCE_SHA256,
           "identity_scope":"ZIP bytes + packaged source stamp comparison; not Git history authentication"}
try:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("고정 source ZIP 한 개만 필요합니다.")
    source_name, raw = next(iter(uploaded.items()))
    if not source_name.lower().endswith(".zip") or len(raw) > 64*1024**2:
        raise ValueError("source ZIP 64MiB 상한")
    actual = hashlib.sha256(raw).hexdigest()
    receipt.update(name=source_name, bytes=len(raw), sha256=actual)
    if actual != EXPECTED_SOURCE_SHA256:
        raise ValueError("source ZIP SHA256 불일치")
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        members = archive.infolist()
        if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
            raise ValueError("소스 해제 상한 초과")
        roots = [PurePosixPath(m.filename).parent.parent for m in members if m.filename.endswith("rewind_m1/diagnostic_smoke.py")]
        if len(roots)!=1:
            raise ValueError("통합 worker가 있는 단일 source ZIP 필요")
        seen = set()
        excluded = {".git",".venv","data","outputs","checkpoints",".local-archive",".aws",".codex",".agents","__pycache__"}
        for m in members:
            p = PurePosixPath(m.filename)
            mode = stat.S_IFMT(m.external_attr>>16)
            if (p.is_absolute() or ".." in p.parts or "\\" in m.filename or not p.parts
                or any(":" in x or x.endswith(("."," ")) for x in p.parts)
                or any(ord(c)<32 for c in m.orig_filename)
                or mode not in (0,stat.S_IFREG,stat.S_IFDIR) or m.flag_bits&1):
                raise ValueError("안전하지 않은 source ZIP")
            key = p.as_posix().casefold().rstrip("/")
            if key in seen:
                raise ValueError("중복 source 경로")
            seen.add(key)
            if any(x in excluded or x==".env" or x.startswith(".env.") for x in p.parts):
                raise ValueError("소스 ZIP에 제외 대상 파일이 있습니다.")
        extract_root = M1_RUN_ROOT/"source"
        extract_root.mkdir()
        archive.extractall(extract_root)
    PROJECT_ROOT = (extract_root/roots[0]).resolve()
    if not PROJECT_ROOT.is_relative_to(extract_root.resolve()):
        raise ValueError("source root 이탈")
    stamp = (PROJECT_ROOT/"SOURCE_REVISION.txt").read_text(encoding="utf-8").strip()
    if stamp != EXPECTED_REVISION:
        raise ValueError("SOURCE_REVISION stamp 불일치")
    SOURCE_OK = True
    receipt.update(status="ready", source_stamp=stamp)
except Exception as exc:
    receipt["error_type"] = type(exc).__name__
finally:
    (M1_RUN_ROOT/"source_upload.json").write_text(json.dumps(receipt,ensure_ascii=False),encoding="utf-8")
    for name in ("uploaded","raw"):
        globals().pop(name,None)
print(receipt)


In [ ]:
# 기존 Colab CPU 환경 확인. 자동 설치·GPU 실행 없음.
import importlib.util, importlib.metadata, platform
environment = {"python":platform.python_version(), "source_ready":SOURCE_OK}
for package, module in (("torch","torch"),("Pillow","PIL")):
    try:
        version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        version = None
    environment[package] = {"present":importlib.util.find_spec(module) is not None, "version":version}
ENV_READY = SOURCE_OK and all(environment[p]["present"] for p in ("torch","Pillow"))
torch_version = environment["torch"]["version"] or ""
match = re.match(r"^(\d+)\.(\d+)", torch_version)
ENV_READY = ENV_READY and bool(match and tuple(map(int,match.groups())) >= (2,3))
environment["ready"] = ENV_READY
(M1_RUN_ROOT/"environment.json").write_text(json.dumps(environment),encoding="utf-8")
print(environment)
if not ENV_READY:
    print("환경/source 준비 실패: 실행하지 말고 결과를 회수해 테스트 세션에 전달하세요. 자동 설치하지 않습니다.")


In [ ]:
# 단일 worker 1회. 로그를 실행 중 각 64KiB로 제한하며 초과 시 종료한다.
import subprocess, threading, time
if globals().get("M1_STARTED",False):
    raise RuntimeError("자동 반복 금지. 이전 결과를 먼저 회수하세요.")
M1_STARTED = True
supervision = {"status":"failed", "timeout_seconds":180, "started":False}
process = None
threads = []
overflow = threading.Event()
pump_error = threading.Event()
pump_errors = []
cleanup_errors = []
started = time.monotonic()
def pump(pipe, path):
    size = 0
    try:
        with path.open("xb") as log:
            while True:
                chunk = pipe.read(4096)
                if not chunk:
                    break
                log.write(chunk[:max(0,65536-size)])
                size += len(chunk)
                if size > 65536:
                    overflow.set()
                    process.kill()
    except BaseException as exc:
        pump_errors.append(type(exc).__name__)
        pump_error.set()
        try:
            process.kill()
        except BaseException as stop_exc:
            cleanup_errors.append(type(stop_exc).__name__)
    finally:
        try:
            pipe.close()
        except BaseException as exc:
            pump_errors.append(type(exc).__name__)
            pump_error.set()
try:
    if not ENV_READY:
        raise RuntimeError("환경/source 준비 실패")
    command = [sys.executable,"-B","-m","rewind_m1.diagnostic_smoke",
               "--output",str(M1_RUN_ROOT/"check"),"--expected-revision",EXPECTED_REVISION]
    supervision["started"] = True
    process = subprocess.Popen(command,cwd=PROJECT_ROOT,stdout=subprocess.PIPE,stderr=subprocess.PIPE,
        env={**os.environ,"PYTHONDONTWRITEBYTECODE":"1","CUDA_VISIBLE_DEVICES":"","OMP_NUM_THREADS":"1","MKL_NUM_THREADS":"1"})
    for pipe,name in ((process.stdout,"stdout.log"),(process.stderr,"stderr.log")):
        thread = threading.Thread(target=pump,args=(pipe,M1_RUN_ROOT/name),daemon=True)
        thread.start()
        threads.append(thread)
    process.wait(timeout=180)
    for thread in threads:
        thread.join(timeout=5)
    supervision["returncode"] = process.returncode
    summary = json.loads((M1_RUN_ROOT/"check/summary.json").read_text(encoding="utf-8"))
    supervision["status"] = "passed" if process.returncode==0 and summary.get("status")=="passed" and not overflow.is_set() else "failed"
except subprocess.TimeoutExpired:
    supervision["status"] = "timeout"
except BaseException as exc:
    supervision.update(status="interrupted" if isinstance(exc,KeyboardInterrupt) else "failed",error_type=type(exc).__name__)
finally:
    try:
        if process is not None and process.poll() is None:
            process.kill()
            process.wait(timeout=5)
    except BaseException as exc:
        cleanup_errors.append(type(exc).__name__)
    for thread in threads:
        try:
            thread.join(timeout=5)
        except BaseException as exc:
            cleanup_errors.append(type(exc).__name__)
    alive = any(thread.is_alive() for thread in threads)
    supervision.update(elapsed_seconds=time.monotonic()-started, log_overflow=overflow.is_set(),
        pump_errors=list(pump_errors), cleanup_errors=list(cleanup_errors), log_threads_alive=alive)
    if overflow.is_set() or pump_error.is_set() or cleanup_errors or alive:
        supervision["status"] = "failed"
    (M1_RUN_ROOT/"supervision.json").write_text(json.dumps(supervision),encoding="utf-8")
print(supervision)
print("성공·실패·timeout 모두 다음 셀로 결과를 회수하세요. 재실행하지 않습니다.")


In [ ]:
# 기존 결과는 재다운로드만 허용. 새 ZIP은 임시 파일에서 완성 후 교체.
MAX_EXPORT = 80*1024**2
result_zip = M1_RUN_ROOT/"local_m1_diagnostic_result.zip"
if result_zip.exists():
    export_report = json.loads((M1_RUN_ROOT/"export_report.json").read_text(encoding="utf-8"))
else:
    entries = [(p,p.relative_to(M1_RUN_ROOT).as_posix()) for p in sorted(M1_RUN_ROOT.rglob("*"))
               if p.is_file() and "source" not in p.relative_to(M1_RUN_ROOT).parts
               and not p.name.endswith(".zip.tmp")
               and p.name not in ("local_m1_diagnostic_result.zip","export_report.json")]
    total = sum(p.stat().st_size for p,_ in entries)
    export_report = {"status":"ready", "original_bytes":total, "excluded":[], "originals_preserved":True}
    if total > MAX_EXPORT:
        export_report.update(status="partial_export", reason="uncompressed export byte limit")
        kept = []
        budget = 512*1024
        for path,name in entries:
            size = path.stat().st_size
            if path.suffix in (".json",".log") and size <= budget:
                kept.append((path,name))
                budget -= size
            else:
                export_report["excluded"].append({"path":name,"size_bytes":size})
        entries = kept
    receipt_path = M1_RUN_ROOT/"export_report.json"
    receipt_path.write_text(json.dumps(export_report),encoding="utf-8")
    entries.append((receipt_path,"export_report.json"))
    with tempfile.NamedTemporaryFile(dir=M1_RUN_ROOT,suffix=".zip.tmp",delete=False) as handle:
        temporary_zip = Path(handle.name)
    try:
        with zipfile.ZipFile(temporary_zip,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
            for path,name in entries:
                bundle.write(path,name)
        if temporary_zip.stat().st_size > MAX_EXPORT:
            export_report.update(status="export_failed",reason="compressed export byte limit")
            export_report["excluded"] = [{"path":name,"size_bytes":path.stat().st_size} for path,name in entries if name!="export_report.json"]
            receipt_path.write_text(json.dumps(export_report),encoding="utf-8")
            with zipfile.ZipFile(temporary_zip,"w",compression=zipfile.ZIP_DEFLATED) as bundle:
                bundle.write(receipt_path,"export_report.json")
        if temporary_zip.stat().st_size > MAX_EXPORT:
            raise ValueError("최소 export 보고서도 ZIP 상한 초과: 원본을 보존하고 중단하세요.")
        os.replace(temporary_zip,result_zip)
    finally:
        temporary_zip.unlink(missing_ok=True)
print({"export":export_report["status"],"result_zip":str(result_zip),"bytes":result_zip.stat().st_size,
       "sha256":hashlib.sha256(result_zip.read_bytes()).hexdigest()})
files.download(str(result_zip))
